In [53]:
# =========================
# 1. IMPORTS
# =========================
from __future__ import annotations

import copy
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

In [54]:
# =========================
# 2. CONFIG
# =========================
PREPROCESSED_ROOT = Path(r"D:/HUP_processed/pipeline_b_car_0p5_120_128")
EXPERIMENT_ROOT = Path(r"D:\hup_all_subjects_with_llm_cohort_sub146")

# These subjects get special LLM holdout runs.
LLM_COHORT_SUBJECTS: List[str] = [
    "sub-HUP070",
    "sub-HUP097",
    "sub-HUP107",
    "sub-HUP148",
    "sub-HUP146",
    "sub-HUP164",
]

# Optional filter. Leave empty to process every subject in PREPROCESSED_ROOT.
SUBJECT_FILTER = ["sub-HUP146"]

# Optional manual override for reserved LLM runs.
MANUAL_LLM_HOLDOUTS: Dict[str, Dict[str, str]] = {}

INCLUDE_TOKENS: List[str] = []
SEED = 42
DEVICE = "cuda"
BATCH_SIZE = 32
NUM_EPOCHS = 35
LR = 3e-4
WEIGHT_DECAY = 3e-4
DROPOUT = 0.30
EARLY_STOPPING_PATIENCE = 5
SMOOTHING_KERNEL = 5

CNN_HIDDEN = 32
EMBED_DIM = 64
NHEAD = 2
NUM_LAYERS = 1
FF_MULT = 4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.5

LLM_UNIFIED_ROOT = Path(r"D:\LLM_unified_HUP")
THRESH_GRID = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

In [55]:
# =========================================================
# 3. UTILS
# =========================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def smooth_probs(probs: np.ndarray, kernel: int = 5) -> np.ndarray:
    if kernel <= 1 or len(probs) == 0:
        return probs
    k = np.ones(kernel, dtype=np.float32) / kernel
    return np.convolve(probs, k, mode="same")


set_seed(SEED)
device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")

In [56]:
# =========================================================
# 4. DATASET
# =========================================================
class WindowDataset(Dataset):
    def __init__(self, X, y_eval, y_train, mask, t_bounds, run_ids):
        self.X = torch.from_numpy(X).float()
        self.y_eval = torch.from_numpy(y_eval).float()
        self.y_train = torch.from_numpy(y_train).float()
        self.mask = torch.from_numpy(mask).bool()
        self.t_bounds = torch.from_numpy(t_bounds).float()
        self.run_ids = np.asarray(run_ids)

    def __len__(self):
        return len(self.y_eval)

    def __getitem__(self, idx):
        return {
            "x": self.X[idx],
            "y_eval": self.y_eval[idx],
            "y_train": self.y_train[idx],
            "mask": self.mask[idx],
            "t_bounds": self.t_bounds[idx],
            "run_id": self.run_ids[idx],
        }


def make_loader(dataset, y_train_np, batch_size: int, train: bool = False):
    if train:
        valid = y_train_np >= 0
        trainable = y_train_np[valid].astype(np.int64)
        counts = np.bincount(trainable, minlength=2) if len(trainable) else np.array([1, 1])
        counts = np.maximum(counts, 1)
        class_weights = 1.0 / counts
        sample_weights = np.ones(len(y_train_np), dtype=np.float32)
        sample_weights[valid] = class_weights[y_train_np[valid].astype(np.int64)]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        return DataLoader(dataset, batch_size=batch_size, sampler=sampler, drop_last=False)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)

In [57]:
# =========================================================
# 5. MODEL
# =========================================================
class TemporalCNNEncoder(nn.Module):
    def __init__(self, hidden: int = 32, embed_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, hidden, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, embed_dim, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(embed_dim), nn.GELU(), nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        B, C, T = x.shape
        x = x.reshape(B * C, 1, T)
        z = self.net(x).squeeze(-1)
        return z.reshape(B, C, -1)


class ChannelTransformerClassifier(nn.Module):
    def __init__(self, embed_dim=64, nhead=2, num_layers=1, ff_mult=4, dropout=0.3, cnn_hidden=32):
        super().__init__()
        self.temporal = TemporalCNNEncoder(hidden=cnn_hidden, embed_dim=embed_dim, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=ff_mult * embed_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.cls = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Dropout(dropout), nn.Linear(embed_dim, 1))

    def forward(self, x, mask):
        z = self.temporal(x)
        B = z.shape[0]
        cls = self.cls.expand(B, -1, -1)
        z = torch.cat([cls, z], dim=1)
        cls_mask = torch.ones((B, 1), dtype=torch.bool, device=mask.device)
        src_key_padding_mask = ~torch.cat([cls_mask, mask], dim=1)
        z = self.transformer(z, src_key_padding_mask=src_key_padding_mask)
        return self.head(z[:, 0]).squeeze(-1)


class BinaryFocalLossIgnore(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.5):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        valid = targets >= 0
        if valid.sum() == 0:
            return logits.sum() * 0.0
        logits = logits[valid]
        targets = targets[valid]
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probas = torch.sigmoid(logits)
        p_t = probas * targets + (1.0 - probas) * (1.0 - targets)
        loss = ((1.0 - p_t) ** self.gamma) * bce
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        return (alpha_t * loss).mean()



In [58]:
# =========================================================
# 6. DATA LOADING
# =========================================================
def load_subject_records(subject_dir: Path) -> List[dict]:
    npz_files = sorted(subject_dir.glob("*.npz"))
    if INCLUDE_TOKENS:
        npz_files = [p for p in npz_files if all(tok.lower() in p.stem.lower() for tok in INCLUDE_TOKENS)]

    records = []
    for npz_path in npz_files:
        with np.load(npz_path, allow_pickle=True) as d:
            y = d["y"].astype(np.int64)
            task = "ictal" if np.any(y == 1) else "interictal"
            records.append({
                "npz_path": npz_path,
                "run_id": npz_path.stem,
                "task": task,
                "X": d["X"].astype(np.float32),
                "y": y,
                "y_train": d["y_train"].astype(np.int64) if "y_train" in d else y.copy(),
                "mask": d["mask"].astype(bool),
                "t_bounds": d["t_bounds"].astype(np.float32),
                "n_windows": int(len(y)),
                "n_ictal": int((y == 1).sum()),
                "n_nonictal": int((y == 0).sum()),
            })
    return records


def pad_records(records: List[dict]) -> dict:
    max_c = max(r["X"].shape[1] for r in records)
    T = records[0]["X"].shape[2]
    Xs, ys, ytrs, masks, tb, run_ids, groups, tasks = [], [], [], [], [], [], [], []

    for gi, r in enumerate(records):
        X = r["X"]
        n, c, _ = X.shape
        Xpad = np.zeros((n, max_c, T), dtype=np.float32)
        Mpad = np.zeros((n, max_c), dtype=bool)
        Xpad[:, :c, :] = X
        Mpad[:, :c] = True

        Xs.append(Xpad)
        ys.append(r["y"])
        ytrs.append(r["y_train"])
        masks.append(Mpad)
        tb.append(r["t_bounds"])
        run_ids.extend([r["run_id"]] * n)
        groups.extend([gi] * n)
        tasks.extend([r["task"]] * n)

    return {
        "X_all": np.concatenate(Xs, axis=0),
        "y_all": np.concatenate(ys, axis=0),
        "y_train_all": np.concatenate(ytrs, axis=0),
        "mask_all": np.concatenate(masks, axis=0),
        "t_bounds_all": np.concatenate(tb, axis=0),
        "run_ids_all": np.asarray(run_ids),
        "groups_all": np.asarray(groups),
        "tasks_all": np.asarray(tasks),
    }


def build_window_dataset(arrays: dict, idx: np.ndarray) -> WindowDataset:
    return WindowDataset(
        arrays["X_all"][idx], arrays["y_all"][idx], arrays["y_train_all"][idx],
        arrays["mask_all"][idx], arrays["t_bounds_all"][idx], arrays["run_ids_all"][idx],
    )

In [59]:
# =========================================================
# 7. SPLITS
# =========================================================
def choose_llm_holdout_runs(subject_name: str, records: List[dict]) -> Dict[str, str]:
    if subject_name in MANUAL_LLM_HOLDOUTS:
        return MANUAL_LLM_HOLDOUTS[subject_name]

    ictal_runs = sorted([r for r in records if r["task"] == "ictal"], key=lambda r: (r["n_ictal"], r["run_id"]))
    interictal_runs = sorted([r for r in records if r["task"] == "interictal"], key=lambda r: (r["n_nonictal"], r["run_id"]))

    if len(ictal_runs) < 2:
        raise RuntimeError(f"{subject_name}: need at least 2 ictal runs for LLM holdout strategy.")
    if len(interictal_runs) < 1:
        raise RuntimeError(f"{subject_name}: need at least 1 interictal run for LLM holdout strategy.")

    return {
        "ictal": ictal_runs[-1]["run_id"],
        "interictal": interictal_runs[-1]["run_id"],
    }


def split_train_vs_llm_holdout(subject_name: str, records: List[dict]) -> Tuple[List[dict], List[dict], Dict[str, str] | None]:
    if subject_name not in LLM_COHORT_SUBJECTS:
        return records, [], None

    chosen = choose_llm_holdout_runs(subject_name, records)
    reserved_ids = {chosen["ictal"], chosen["interictal"]}
    train_records = [r for r in records if r["run_id"] not in reserved_ids]
    reserved_records = [r for r in records if r["run_id"] in reserved_ids]

    if len(train_records) < 2:
        raise RuntimeError(f"{subject_name}: too few training runs remain after reserving LLM holdouts.")

    return train_records, reserved_records, chosen


def make_group_loso_splits(groups: np.ndarray, tasks_all: np.ndarray):
    uniq = np.unique(groups)
    splits = []
    for g in uniq:
        test_idx = np.where(groups == g)[0]
        if str(tasks_all[test_idx[0]]) != "ictal":
            continue
        trainval_idx = np.where(groups != g)[0]
        remaining_groups = np.unique(groups[trainval_idx])
        if len(remaining_groups) >= 2:
            gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED + int(g))
            rel_train, rel_val = next(gss.split(np.zeros(len(trainval_idx)), groups=groups[trainval_idx]))
            train_idx = trainval_idx[rel_train]
            val_idx = trainval_idx[rel_val]
        else:
            cut = max(1, int(0.8 * len(trainval_idx)))
            train_idx = trainval_idx[:cut]
            val_idx = trainval_idx[cut:]
        splits.append((int(g), train_idx, val_idx, test_idx))
    return splits


def make_final_train_val_split(arrays: dict) -> Tuple[np.ndarray, np.ndarray]:
    groups = arrays["groups_all"]
    tasks = arrays["tasks_all"]
    uniq = np.unique(groups)
    ictal_groups = [g for g in uniq if str(tasks[np.where(groups == g)[0][0]]) == "ictal"]
    val_group = ictal_groups[-1] if len(ictal_groups) >= 1 else uniq[-1]
    val_idx = np.where(groups == val_group)[0]
    train_idx = np.where(groups != val_group)[0]
    return train_idx, val_idx

In [61]:
# =========================================================
# 8. TRAIN / EVAL
# =========================================================
def run_epoch(model, loader, optimizer=None, criterion=None, threshold=0.5):
    is_train = optimizer is not None
    model.train(is_train)

    all_probs, all_targets, all_run_ids, all_t_bounds = [], [], [], []
    losses = []

    for batch in loader:
        x = batch["x"].to(device)
        mask = batch["mask"].to(device)
        y_eval = batch["y_eval"].to(device)
        y_train = batch["y_train"].to(device)
        logits = model(x, mask)

        loss = criterion(logits, y_train) if criterion is not None else torch.tensor(0.0, device=device)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        losses.append(float(loss.item()))
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        all_probs.append(probs)
        all_targets.append(y_eval.detach().cpu().numpy())
        all_run_ids.extend(batch["run_id"])
        all_t_bounds.append(batch["t_bounds"].detach().cpu().numpy())

    probs = np.concatenate(all_probs) if all_probs else np.array([])
    targets = np.concatenate(all_targets).astype(int) if all_targets else np.array([], dtype=int)
    t_bounds = np.concatenate(all_t_bounds, axis=0) if all_t_bounds else np.empty((0, 2), dtype=np.float32)

    preds = (probs >= threshold).astype(int) if len(probs) else np.array([], dtype=int)

    return {
        "loss": float(np.mean(losses)) if losses else np.nan,
        "accuracy": accuracy_score(targets, preds) if len(targets) else np.nan,
        "balanced_accuracy": balanced_accuracy_score(targets, preds) if len(np.unique(targets)) > 1 else np.nan,
        "precision": precision_score(targets, preds, zero_division=0) if len(targets) else np.nan,
        "recall": recall_score(targets, preds, zero_division=0) if len(targets) else np.nan,
        "f1": f1_score(targets, preds, zero_division=0) if len(targets) else np.nan,
        "auroc": roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else np.nan,
        "auprc": average_precision_score(targets, probs) if len(np.unique(targets)) > 1 else np.nan,
        "targets": targets,
        "probs": probs,
        "preds": preds,
        "run_ids": np.asarray(all_run_ids),
        "t_bounds": t_bounds,
    }


def select_best_threshold(y_true: np.ndarray, probs_s: np.ndarray, grid: np.ndarray):
    """
    Pick threshold by balanced accuracy on smoothed validation probabilities.
    Tie-breakers: F1, then precision.
    """
    if len(np.unique(y_true)) < 2:
        return 0.5, {
            "threshold": 0.5,
            "balanced_accuracy": np.nan,
            "f1": np.nan,
            "precision": np.nan,
            "recall": np.nan,
        }

    rows = []
    for thr in grid:
        pred = (probs_s >= thr).astype(int)
        rows.append({
            "threshold": float(thr),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
            "f1": f1_score(y_true, pred, zero_division=0),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
        })

    df = pd.DataFrame(rows).sort_values(
        ["balanced_accuracy", "f1", "precision", "threshold"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

    best = df.iloc[0].to_dict()
    return float(best["threshold"]), best


def fit_model(train_ds, val_ds):
    train_loader = make_loader(train_ds, train_ds.y_train.numpy(), BATCH_SIZE, train=True)
    val_loader = make_loader(val_ds, val_ds.y_train.numpy(), BATCH_SIZE, train=False)

    model = ChannelTransformerClassifier(
        embed_dim=EMBED_DIM, nhead=NHEAD, num_layers=NUM_LAYERS,
        ff_mult=FF_MULT, dropout=DROPOUT, cnn_hidden=CNN_HIDDEN,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = BinaryFocalLossIgnore(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)

    best_state, best_thr, best_score = None, 0.5, -np.inf
    history, patience = [], 0

    for epoch in range(1, NUM_EPOCHS + 1):
        train_out = run_epoch(model, train_loader, optimizer=optimizer, criterion=criterion)
        val_out = run_epoch(model, val_loader, optimizer=None, criterion=criterion)

        if len(np.unique(val_out["targets"])) > 1:
            val_probs_s = smooth_probs(val_out["probs"], SMOOTHING_KERNEL)
            cur_thr, thr_info = select_best_threshold(val_out["targets"], val_probs_s, THRESH_GRID)
            cur_score = float(thr_info["balanced_accuracy"])
        else:
            cur_thr = 0.5
            cur_score = float(val_out["balanced_accuracy"] if not np.isnan(val_out["balanced_accuracy"]) else 0.0)
            thr_info = {
                "threshold": cur_thr,
                "balanced_accuracy": cur_score,
                "f1": float(val_out["f1"] if not np.isnan(val_out["f1"]) else 0.0),
                "precision": float(val_out["precision"] if not np.isnan(val_out["precision"]) else 0.0),
                "recall": float(val_out["recall"] if not np.isnan(val_out["recall"]) else 0.0),
            }

        history.append({
            "epoch": epoch,
            "train_loss": train_out["loss"],
            "train_f1": train_out["f1"],
            "val_loss": val_out["loss"],
            "val_best_thr": cur_thr,
            "val_balanced_acc_smooth": thr_info["balanced_accuracy"],
            "val_f1_smooth": thr_info["f1"],
            "val_precision_smooth": thr_info["precision"],
            "val_recall_smooth": thr_info["recall"],
        })

        if cur_score > best_score:
            best_score = cur_score
            best_thr = cur_thr
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1

        if patience >= EARLY_STOPPING_PATIENCE:
            break

    if best_state is None:
        raise RuntimeError("Training failed: no best model state.")

    model.load_state_dict(best_state)
    return model, best_thr, pd.DataFrame(history)


def evaluate_dataset(model, thr, ds):
    loader = make_loader(ds, ds.y_train.numpy(), BATCH_SIZE, train=False)
    out = run_epoch(model, loader, optimizer=None, criterion=None)

    probs_raw = out["probs"]
    probs_s = smooth_probs(probs_raw, SMOOTHING_KERNEL)
    preds = (probs_s >= thr).astype(int)
    y = out["targets"]

    cm = confusion_matrix(y, preds, labels=[0, 1])

    return {
        "acc": accuracy_score(y, preds),
        "balanced_acc": balanced_accuracy_score(y, preds) if len(np.unique(y)) > 1 else np.nan,
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "auroc": roc_auc_score(y, probs_s) if len(np.unique(y)) > 1 else np.nan,
        "auprc": average_precision_score(y, probs_s) if len(np.unique(y)) > 1 else np.nan,
        "n_test": len(y),
        "n_ictal": int((y == 1).sum()),
        "n_nonictal": int((y == 0).sum()),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "y_true": y,
        "prob_raw": probs_raw,
        "prob_smooth": probs_s,
        "pred": preds,
        "run_ids": out["run_ids"],
        "t_bounds": out["t_bounds"],
        "threshold_used": float(thr),
    }

In [62]:
# =========================================================
# 9. LLM JSON EXPORT HELPERS
# =========================================================
def summarize_positive_segments(pred_df: pd.DataFrame) -> List[dict]:
    """
    Merge consecutive positive windows into coarse candidate ictal segments.
    """
    if pred_df.empty:
        return []

    pos = pred_df[pred_df["pred"] == 1].copy()
    if pos.empty:
        return []

    pos = pos.sort_values("start_sec").reset_index(drop=True)

    segments = []
    cur_start = float(pos.loc[0, "start_sec"])
    cur_end = float(pos.loc[0, "end_sec"])
    cur_probs = [float(pos.loc[0, "prob_smooth"])]
    cur_n = 1

    for i in range(1, len(pos)):
        row = pos.loc[i]
        start_i = float(row["start_sec"])
        end_i = float(row["end_sec"])
        prob_i = float(row["prob_smooth"])

        # merge if overlapping or directly adjacent
        if start_i <= cur_end + 1e-6:
            cur_end = max(cur_end, end_i)
            cur_probs.append(prob_i)
            cur_n += 1
        else:
            segments.append({
                "start_sec": cur_start,
                "end_sec": cur_end,
                "duration_sec": cur_end - cur_start,
                "n_positive_windows": cur_n,
                "mean_prob_smooth": float(np.mean(cur_probs)),
                "peak_prob_smooth": float(np.max(cur_probs)),
            })
            cur_start = start_i
            cur_end = end_i
            cur_probs = [prob_i]
            cur_n = 1

    segments.append({
        "start_sec": cur_start,
        "end_sec": cur_end,
        "duration_sec": cur_end - cur_start,
        "n_positive_windows": cur_n,
        "mean_prob_smooth": float(np.mean(cur_probs)),
        "peak_prob_smooth": float(np.max(cur_probs)),
    })
    return segments


def build_run_level_prediction_df(rev: dict) -> pd.DataFrame:
    df = pd.DataFrame({
        "run_id": rev["run_ids"],
        "y_true": rev["y_true"],
        "prob_raw": rev["prob_raw"],
        "prob_smooth": rev["prob_smooth"],
        "pred": rev["pred"],
        "start_sec": rev["t_bounds"][:, 0],
        "end_sec": rev["t_bounds"][:, 1],
    })
    return df.sort_values(["run_id", "start_sec", "end_sec"]).reset_index(drop=True)


def build_reserved_run_json(
    subject_id: str,
    run_id: str,
    run_task: str,
    run_df: pd.DataFrame,
    threshold_used: float,
) -> dict:
    positive_segments = summarize_positive_segments(run_df)

    pred_pos = int((run_df["pred"] == 1).sum())
    true_pos = int((run_df["y_true"] == 1).sum())

    out = {
        "subject_id": subject_id,
        "module": "seizure_detection",
        "dataset": "HUP",
        "run_id": run_id,
        "task": run_task,
        "window_level_summary": {
            "n_windows": int(len(run_df)),
            "n_true_ictal_windows": true_pos,
            "n_true_nonictal_windows": int((run_df["y_true"] == 0).sum()),
            "n_predicted_positive_windows": pred_pos,
            "fraction_predicted_positive": float(pred_pos / len(run_df)) if len(run_df) else 0.0,
            "mean_prob_smooth": float(run_df["prob_smooth"].mean()) if len(run_df) else 0.0,
            "peak_prob_smooth": float(run_df["prob_smooth"].max()) if len(run_df) else 0.0,
            "threshold_used": float(threshold_used),
        },
        "run_level_interpretation": {
            "predicted_ictal_present": bool(pred_pos > 0),
            "first_positive_start_sec": float(run_df.loc[run_df["pred"] == 1, "start_sec"].min()) if pred_pos > 0 else None,
            "last_positive_end_sec": float(run_df.loc[run_df["pred"] == 1, "end_sec"].max()) if pred_pos > 0 else None,
            "candidate_segments": positive_segments,
        },
        "window_predictions": [
            {
                "start_sec": float(r.start_sec),
                "end_sec": float(r.end_sec),
                "y_true": int(r.y_true),
                "prob_raw": float(r.prob_raw),
                "prob_smooth": float(r.prob_smooth),
                "pred": int(r.pred),
            }
            for r in run_df.itertuples(index=False)
        ],
        "llm_notes": [
            "Use only information explicitly present in this JSON.",
            "Do not infer channel-level seizure onset from this file because channel-wise outputs are not included.",
            "Candidate segments are derived from consecutive positive windows after smoothing and thresholding.",
            "If positive windows are diffuse or frequent in interictal runs, state uncertainty and possible false positives."
        ],
    }
    return out


def export_reserved_run_jsons(
    subject_id: str,
    chosen: Dict[str, str],
    rev: dict,
):
    llm_seizure_dir = LLM_UNIFIED_ROOT / subject_id / "seizure"
    llm_seizure_dir.mkdir(parents=True, exist_ok=True)

    pred_df = build_run_level_prediction_df(rev)

    run_task_map = {
        chosen["ictal"]: "ictal",
        chosen["interictal"]: "interictal",
    }

    for run_id, run_task in run_task_map.items():
        run_df = pred_df[pred_df["run_id"] == run_id].copy().reset_index(drop=True)
        if run_df.empty:
            print(f"[WARN] No reserved predictions found for {run_id}")
            continue

        run_json = build_reserved_run_json(
            subject_id=subject_id,
            run_id=run_id,
            run_task=run_task,
            run_df=run_df,
            threshold_used=rev["threshold_used"],
        )

        out_name = f"{run_id}_seizure_summary_for_llm.json"
        out_path = llm_seizure_dir / out_name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(run_json, f, indent=2)

        print(f"Saved LLM seizure JSON: {out_path}")

In [63]:
# =========================================================
# 10. MAIN
# =========================================================
def find_subject_dirs(root: Path) -> List[Path]:
    subs = [p for p in sorted(root.iterdir()) if p.is_dir() and p.name.startswith("sub-")]
    if SUBJECT_FILTER:
        subs = [p for p in subs if p.name in SUBJECT_FILTER]
    return subs


def main():
    EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
    subject_dirs = find_subject_dirs(PREPROCESSED_ROOT)
    if not subject_dirs:
        raise RuntimeError(f"No subjects found in {PREPROCESSED_ROOT}")

    all_internal_rows = []
    all_reserved_rows = []
    cohort_manifest = []

    for subject_dir in subject_dirs:
        subject_name = subject_dir.name
        print("\n" + "#" * 120)
        print(f"SUBJECT: {subject_name}")
        print("#" * 120)

        records = load_subject_records(subject_dir)
        if len(records) < 2:
            print(f"Skipping {subject_name}: fewer than 2 runs.")
            continue

        train_records, reserved_records, chosen = split_train_vs_llm_holdout(subject_name, records)
        out_dir = EXPERIMENT_ROOT / subject_name
        out_dir.mkdir(parents=True, exist_ok=True)

        if chosen is not None:
            cohort_manifest.append({
                "subject_id": subject_name,
                "reserved_ictal_run": chosen["ictal"],
                "reserved_interictal_run": chosen["interictal"],
                "n_train_runs_remaining": len(train_records),
                "n_reserved_runs": len(reserved_records),
            })
            with open(out_dir / "llm_reserved_runs.json", "w", encoding="utf-8") as f:
                json.dump(chosen, f, indent=2)

        arrays = pad_records(train_records)
        splits = make_group_loso_splits(arrays["groups_all"], arrays["tasks_all"])
        if not splits:
            print(f"Skipping {subject_name}: no valid ictal-heldout folds.")
            continue

        internal_rows = []
        for fold_id, train_idx, val_idx, test_idx in splits:
            heldout_run = np.unique(arrays["run_ids_all"][test_idx]).tolist()
            print("=" * 100)
            print(f"{subject_name} | INTERNAL FOLD {fold_id:02d} | HELD-OUT: {heldout_run}")
            print("=" * 100)

            train_ds = build_window_dataset(arrays, train_idx)
            val_ds = build_window_dataset(arrays, val_idx)
            test_ds = build_window_dataset(arrays, test_idx)
            model, thr, hist = fit_model(train_ds, val_ds)
            ev = evaluate_dataset(model, thr, test_ds)

            row = {
                "subject_id": subject_name,
                "fold_id": fold_id,
                "held_out_run": heldout_run[0],
                "test_acc": ev["acc"],
                "test_balanced_acc": ev["balanced_acc"],
                "test_precision": ev["precision"],
                "test_recall": ev["recall"],
                "test_f1": ev["f1"],
                "test_auroc": ev["auroc"],
                "test_auprc": ev["auprc"],
                "n_test": ev["n_test"],
                "n_test_ictal": ev["n_ictal"],
                "n_test_nonictal": ev["n_nonictal"],
                "tn": ev["tn"], "fp": ev["fp"], "fn": ev["fn"], "tp": ev["tp"],
            }
            internal_rows.append(row)

            fold_dir = out_dir / f"internal_fold_{fold_id:02d}"
            fold_dir.mkdir(parents=True, exist_ok=True)
            hist.to_csv(fold_dir / "training_history.csv", index=False)
            pd.DataFrame({
                "run_id": ev["run_ids"],
                "y_true": ev["y_true"],
                "prob_smooth": ev["prob_smooth"],
                "pred": ev["pred"],
            }).to_csv(fold_dir / "test_predictions.csv", index=False)

        df_internal = pd.DataFrame(internal_rows).sort_values("fold_id")
        df_internal.to_csv(out_dir / "internal_cv_results.csv", index=False)
        all_internal_rows.append(df_internal)

        # For LLM-cohort subjects, also train final model and run on reserved runs.
        # For LLM-cohort subjects, also train final model and run on reserved runs.
        if reserved_records:
            final_train_idx, final_val_idx = make_final_train_val_split(arrays)
            final_train_ds = build_window_dataset(arrays, final_train_idx)
            final_val_ds = build_window_dataset(arrays, final_val_idx)
        
            final_model, final_thr, final_hist = fit_model(final_train_ds, final_val_ds)
            final_hist.to_csv(out_dir / "final_training_history.csv", index=False)
            torch.save(final_model.state_dict(), out_dir / "final_model.pt")
        
            reserved_arrays = pad_records(reserved_records)
            reserved_idx = np.arange(len(reserved_arrays["y_all"]))
            reserved_ds = build_window_dataset(reserved_arrays, reserved_idx)
            rev = evaluate_dataset(final_model, final_thr, reserved_ds)
        
            reserved_pred_df = pd.DataFrame({
                "run_id": rev["run_ids"],
                "start_sec": rev["t_bounds"][:, 0],
                "end_sec": rev["t_bounds"][:, 1],
                "y_true": rev["y_true"],
                "prob_raw": rev["prob_raw"],
                "prob_smooth": rev["prob_smooth"],
                "pred": rev["pred"],
            }).sort_values(["run_id", "start_sec"]).reset_index(drop=True)
        
            reserved_pred_df.to_csv(out_dir / "llm_reserved_predictions.csv", index=False)
        
            # Export one JSON per reserved run for LLM use
            export_reserved_run_jsons(
                subject_id=subject_name,
                chosen=chosen,
                rev=rev,
            )
        
            all_reserved_rows.append({
                "subject_id": subject_name,
                "reserved_ictal_run": chosen["ictal"],
                "reserved_interictal_run": chosen["interictal"],
                "reserved_threshold_used": rev["threshold_used"],
                "reserved_acc": rev["acc"],
                "reserved_balanced_acc": rev["balanced_acc"],
                "reserved_precision": rev["precision"],
                "reserved_recall": rev["recall"],
                "reserved_f1": rev["f1"],
                "reserved_auroc": rev["auroc"],
                "reserved_auprc": rev["auprc"],
                "reserved_n_test": rev["n_test"],
                "reserved_n_ictal": rev["n_ictal"],
                "reserved_n_nonictal": rev["n_nonictal"],
                "tn": rev["tn"],
                "fp": rev["fp"],
                "fn": rev["fn"],
                "tp": rev["tp"],
            })

    if cohort_manifest:
        pd.DataFrame(cohort_manifest).to_csv(EXPERIMENT_ROOT / "llm_cohort_manifest.csv", index=False)
    if all_internal_rows:
        pd.concat(all_internal_rows, ignore_index=True).to_csv(EXPERIMENT_ROOT / "all_internal_cv_results.csv", index=False)
    if all_reserved_rows:
        pd.DataFrame(all_reserved_rows).to_csv(EXPERIMENT_ROOT / "all_llm_reserved_results.csv", index=False)

    print("\nSaved outputs to:", EXPERIMENT_ROOT)


if __name__ == "__main__":
    main()



########################################################################################################################
SUBJECT: sub-HUP146
########################################################################################################################
sub-HUP146 | INTERNAL FOLD 00 | HELD-OUT: ['sub-HUP146_ses-presurgery_task-ictal_acq-seeg_run-01']


C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


sub-HUP146 | INTERNAL FOLD 01 | HELD-OUT: ['sub-HUP146_ses-presurgery_task-ictal_acq-seeg_run-02']


C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Saved LLM seizure JSON: D:\LLM_unified_HUP\sub-HUP146\seizure\sub-HUP146_ses-presurgery_task-ictal_acq-seeg_run-03_seizure_summary_for_llm.json
Saved LLM seizure JSON: D:\LLM_unified_HUP\sub-HUP146\seizure\sub-HUP146_ses-presurgery_task-interictal_acq-seeg_run-02_seizure_summary_for_llm.json

Saved outputs to: D:\hup_all_subjects_with_llm_cohort_sub146
